# RVQ Tokenizer Sweep — Filter Before You Train a BLM

**Goal:** train ~10 candidate Residual Vector Quantization (RVQ) configs on
the spatio-temporal embeddings, score each on tokenizer-quality checks alone
(no BLM training involved), and narrow down to the 2-3 configs actually worth
taking forward to BLM training.

**Why filter first instead of training a BLM per config:** most of what
determines whether a BLM will work is decided at the tokenizer stage — a
collapsed codebook, a vocabulary that's private to one subject, or reconstruction
that barely beats a trivial baseline will sink a BLM regardless of how it's
trained. All of those are checkable directly from the tokenizer's own output,
in seconds, without training anything downstream. Training a full BLM per
config to discover the same thing would be the slowest possible way to get
information this notebook gets almost for free.

**Inputs (same as the clustering notebook):**
- `/content/drive/MyDrive/ds003844_processed/spatio_temporal_embeddings.pt`
- `/content/drive/MyDrive/ds003844_processed/spatio_temporal_metadata.pkl`

**Optional input (for the comparison table at the end):**
- `/content/drive/MyDrive/ds003844_processed/token_discovery/token_discovery_results.pkl`
  (from `token_discovery.ipynb` — if missing, the GMM/HDBSCAN comparison rows
  are just skipped, everything else still runs)

**What this notebook does:**
1. Load embeddings, split by *recording* into train/val (not by window — see Section 1)
2. Define a small RVQ model (encoder + residual quantizer + decoder)
3. Train ~10 configs (varying codebook size / number of levels), each checkpointed independently
4. Score every config: reconstruction MSE, codebook utilization, cross-subject
   recurrence, cross-session recurrence, transition entropy
5. **Funnel**: apply pass/fail thresholds to eliminate obviously broken configs
6. Rank survivors and hand off a shortlist (2-3 configs) — these are the ones
   worth spending BLM training time on


## 0. Setup

In [ ]:
!pip -q install umap-learn scikit-learn --upgrade
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import pickle
import re
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PROCESSED_DIR = "/content/drive/MyDrive/ds003844_processed"
EMBED_PATH = os.path.join(PROCESSED_DIR, "spatio_temporal_embeddings.pt")
META_PATH = os.path.join(PROCESSED_DIR, "spatio_temporal_metadata.pkl")
TOKEN_DISCOVERY_PATH = os.path.join(PROCESSED_DIR, "token_discovery", "token_discovery_results.pkl")
RVQ_DIR = os.path.join(PROCESSED_DIR, "rvq_sweep")
os.makedirs(RVQ_DIR, exist_ok=True)

print("Embeddings path exists:", os.path.exists(EMBED_PATH))
print("Metadata path exists:", os.path.exists(META_PATH))
print("Prior token_discovery results found:", os.path.exists(TOKEN_DISCOVERY_PATH))


## 1. Load data, parse metadata, split by recording

Same filename parsing and same recording-level (not window-level) train/val
split logic as before — splitting by window would leak temporal context
between train and val, since adjacent windows overlap 50%. Stratified by
subject so every one of the 5 subjects appears in both splits.


In [ ]:
final_embeddings = torch.load(EMBED_PATH, map_location="cpu")
print("Flat embeddings shape:", final_embeddings.shape)

with open(META_PATH, "rb") as f:
    metadata = pickle.load(f)

sequence_lengths = metadata["sequence_lengths"]
recording_names = metadata["recording_names"]
channel_counts = metadata["channel_counts"]

assert sum(sequence_lengths) == final_embeddings.shape[0], (
    "Sum of sequence_lengths does not match embeddings row count — "
    "check spatio_temporal_embeddings.pt and spatio_temporal_metadata.pkl "
    "come from the same Notebook 3 run."
)
print("Recordings:", len(recording_names))
print("Total windows:", final_embeddings.shape[0])


In [ ]:
SUB_RE = re.compile(r"(sub-[A-Za-z0-9]+)")
SES_RE = re.compile(r"(ses-[A-Za-z0-9]+)")
SITUATION_RE = re.compile(r"SITUATION(\d)")

def parse_name(name):
    sub_m = SUB_RE.search(name)
    ses_m = SES_RE.search(name)
    sit_m = SITUATION_RE.search(name)
    subject = sub_m.group(1) if sub_m else "unknown_subject"
    session = ses_m.group(1) if ses_m else "unknown_session"
    situation_num = sit_m.group(1) if sit_m else None
    stage_map = {"1": "PRE", "2": "INTER", "3": "POST", "4": "POST+"}
    stage = stage_map.get(situation_num, "unknown_stage")
    return subject, session, stage

parsed = [parse_name(n) for n in recording_names]
subjects_per_rec = [p[0] for p in parsed]
sessions_per_rec = [p[1] for p in parsed]
stages_per_rec = [p[2] for p in parsed]
print("Unique subjects:", sorted(set(subjects_per_rec)))


In [ ]:
rng = np.random.RandomState(42)
subject_to_recs = defaultdict(list)
for i, subj in enumerate(subjects_per_rec):
    subject_to_recs[subj].append(i)

train_rec_idx, val_rec_idx = [], []
VAL_FRACTION = 0.2
for subj, rec_indices in subject_to_recs.items():
    rec_indices = rec_indices.copy()
    rng.shuffle(rec_indices)
    n_val = max(1, int(round(len(rec_indices) * VAL_FRACTION))) if len(rec_indices) > 1 else 0
    val_rec_idx += rec_indices[:n_val]
    train_rec_idx += rec_indices[n_val:]

train_rec_idx = sorted(train_rec_idx)
val_rec_idx = sorted(val_rec_idx)
print(f"Train recordings: {len(train_rec_idx)} | Val recordings: {len(val_rec_idx)}")

def windows_for_recordings(rec_indices, seq_lengths):
    offsets = np.cumsum([0] + list(seq_lengths))
    rows = []
    for rec_idx in rec_indices:
        rows.extend(range(offsets[rec_idx], offsets[rec_idx + 1]))
    return np.array(rows, dtype=np.int64)

train_window_idx = windows_for_recordings(train_rec_idx, sequence_lengths)
val_window_idx = windows_for_recordings(val_rec_idx, sequence_lengths)
print(f"Train windows: {len(train_window_idx)} | Val windows: {len(val_window_idx)}")

train_X = final_embeddings[train_window_idx].to(device)
val_X = final_embeddings[val_window_idx].to(device)

def per_window_metadata(seq_lengths, subjects, sessions, stages):
    out_subjects, out_sessions, out_stages = [], [], []
    for rec_idx, length in enumerate(seq_lengths):
        out_subjects += [subjects[rec_idx]] * length
        out_sessions += [sessions[rec_idx]] * length
        out_stages += [stages[rec_idx]] * length
    return np.array(out_subjects), np.array(out_sessions), np.array(out_stages)

window_subjects, window_sessions, window_stages = per_window_metadata(
    sequence_lengths, subjects_per_rec, sessions_per_rec, stages_per_rec
)


## 2. RVQ model

Small, matched to the style of your existing `GraphAutoencoder`
(`nn.Sequential` + LeakyReLU blocks). Same architecture across all configs —
only `code_dim`, `codebook_size`, and `n_levels` vary between sweep entries,
so any difference in results is attributable to the quantization setup, not
to an unrelated architecture change.


In [ ]:
class ResidualQuantizer(nn.Module):
    def __init__(self, code_dim, codebook_size, n_levels, commitment_cost=0.25):
        super().__init__()
        self.n_levels = n_levels
        self.codebook_size = codebook_size
        self.commitment_cost = commitment_cost
        self.codebooks = nn.ModuleList([
            nn.Embedding(codebook_size, code_dim) for _ in range(n_levels)
        ])
        for cb in self.codebooks:
            nn.init.uniform_(cb.weight, -1.0 / codebook_size, 1.0 / codebook_size)

    def forward(self, z):
        residual = z
        quantized_total = torch.zeros_like(z)
        indices_per_level = []
        commitment_loss = 0.0

        for level in range(self.n_levels):
            codebook = self.codebooks[level].weight
            dist = (
                residual.pow(2).sum(1, keepdim=True)
                - 2 * residual @ codebook.t()
                + codebook.pow(2).sum(1)
            )
            indices = dist.argmin(dim=1)
            quantized = self.codebooks[level](indices)

            commitment_loss = commitment_loss + F.mse_loss(quantized.detach(), residual) \
                + self.commitment_cost * F.mse_loss(quantized, residual.detach())

            quantized_st = residual + (quantized - residual).detach()
            quantized_total = quantized_total + quantized_st
            residual = residual - quantized
            indices_per_level.append(indices)

        indices_per_level = torch.stack(indices_per_level, dim=1)
        return quantized_total, indices_per_level, commitment_loss / self.n_levels


class RVQModel(nn.Module):
    def __init__(self, input_dim=128, code_dim=32, codebook_size=64, n_levels=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, code_dim),
        )
        self.quantizer = ResidualQuantizer(code_dim, codebook_size, n_levels)
        self.decoder = nn.Sequential(
            nn.Linear(code_dim, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        z_q, indices, commitment_loss = self.quantizer(z)
        recon = self.decoder(z_q)
        return recon, indices, commitment_loss


def combine_indices(indices_per_level, codebook_size):
    """Collapse (n_levels) index tuple into one flat token ID, base-codebook_size
    positional encoding — matches the flat-label convention used throughout
    the clustering notebook (gmm_labels/hdbscan_labels)."""
    n_levels = indices_per_level.shape[1]
    flat = torch.zeros(indices_per_level.shape[0], dtype=torch.long, device=indices_per_level.device)
    for level in range(n_levels):
        flat = flat * codebook_size + indices_per_level[:, level]
    return flat


## 3. Define the config sweep

~10 configs spanning small/medium/large per-level codebook size and 2 vs 3
residual levels. Effective vocabulary size is `codebook_size ** n_levels` —
note this grows fast, so the larger configs are included specifically to see
*where* the small-data ceiling is, not because we expect them to win.


In [ ]:
SWEEP_CONFIGS = [
    {"name": "cb16_L1", "code_dim": 32, "codebook_size": 16,  "n_levels": 1},
    {"name": "cb32_L1", "code_dim": 32, "codebook_size": 32,  "n_levels": 1},
    {"name": "cb64_L1", "code_dim": 32, "codebook_size": 64,  "n_levels": 1},
    {"name": "cb8_L2",  "code_dim": 32, "codebook_size": 8,   "n_levels": 2},
    {"name": "cb16_L2", "code_dim": 32, "codebook_size": 16,  "n_levels": 2},
    {"name": "cb32_L2", "code_dim": 32, "codebook_size": 32,  "n_levels": 2},
    {"name": "cb64_L2", "code_dim": 32, "codebook_size": 64,  "n_levels": 2},
    {"name": "cb8_L3",  "code_dim": 32, "codebook_size": 8,   "n_levels": 3},
    {"name": "cb16_L3", "code_dim": 32, "codebook_size": 16,  "n_levels": 3},
    {"name": "cb32_L3", "code_dim": 32, "codebook_size": 32,  "n_levels": 3},
]

for cfg in SWEEP_CONFIGS:
    vocab = cfg["codebook_size"] ** cfg["n_levels"]
    print(f"{cfg['name']:<10} codebook_size={cfg['codebook_size']:<4} "
          f"n_levels={cfg['n_levels']} -> effective vocab = {vocab}")


## 4. Train every config

Each config trains independently and is checkpointed to Drive as it finishes,
so a Colab disconnect partway through the sweep doesn't lose completed
configs — rerunning this cell skips configs whose checkpoint already exists.
150 epochs / batch size 32, matching Notebook 2's convention.


In [ ]:
BATCH_SIZE = 32
EPOCHS = 150
LR = 1e-3

def train_one_config(cfg):
    ckpt_path = os.path.join(RVQ_DIR, f"{cfg['name']}.pt")
    if os.path.exists(ckpt_path):
        print(f"[{cfg['name']}] checkpoint already exists, skipping training")
        return torch.load(ckpt_path, map_location=device)

    model = RVQModel(
        input_dim=128, code_dim=cfg["code_dim"],
        codebook_size=cfg["codebook_size"], n_levels=cfg["n_levels"],
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    n_train = train_X.shape[0]
    best_val_loss = float("inf")
    best_state = None
    train_losses, val_losses = [], []

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=device)
        epoch_loss = 0.0
        for start in range(0, n_train, BATCH_SIZE):
            batch_idx = perm[start:start + BATCH_SIZE]
            batch = train_X[batch_idx]
            recon, indices, commitment_loss = model(batch)
            loss = F.mse_loss(recon, batch) + commitment_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * batch.shape[0]
        epoch_loss /= n_train
        train_losses.append(epoch_loss)

        model.eval()
        with torch.no_grad():
            val_recon, val_indices, val_commitment = model(val_X)
            val_loss = (F.mse_loss(val_recon, val_X) + val_commitment).item()
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}

    elapsed = time.time() - t0
    print(f"[{cfg['name']}] done in {elapsed:.0f}s | best_val_loss={best_val_loss:.5f}")

    result = {
        "config": cfg,
        "state_dict": best_state,
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }
    torch.save(result, ckpt_path)
    return result

sweep_results = {}
for cfg in SWEEP_CONFIGS:
    sweep_results[cfg["name"]] = train_one_config(cfg)


## 5. Score every config — tokenizer-quality checks only, no BLM

For each trained config: reload the best checkpoint, run it over every
window, and compute the same battery used in `token_discovery.ipynb` —
codebook utilization, cross-subject recurrence, cross-session recurrence,
transition entropy — plus held-out reconstruction MSE, which only RVQ
(not GMM/HDBSCAN) can provide.


In [ ]:
def session_letter(session_str):
    m = re.search(r"SITUATION\d([A-Za-z]?)", session_str)
    return m.group(1) if m and m.group(1) else None

rec_session_letters = [session_letter(s) for s in sessions_per_rec]
stage_groups = defaultdict(list)
for rec_idx, (subj, stage, letter) in enumerate(zip(subjects_per_rec, stages_per_rec, rec_session_letters)):
    if letter is not None:
        stage_groups[(subj, stage)].append((letter, rec_idx))
comparable_groups = {k: v for k, v in stage_groups.items() if len(v) > 1}

def token_distribution_for_recording(rec_idx, seq_lengths, labels):
    start = sum(seq_lengths[:rec_idx])
    end = start + seq_lengths[rec_idx]
    rec_labels = labels[start:end]
    counts = Counter(rec_labels.tolist())
    total = sum(counts.values())
    if total == 0:
        return {}
    return {k: v / total for k, v in counts.items()}

def distribution_overlap(dist_a, dist_b):
    keys = set(dist_a) | set(dist_b)
    return sum(min(dist_a.get(k, 0), dist_b.get(k, 0)) for k in keys)

def transition_entropy_for_labels(labels, seq_lengths):
    unique_tokens = sorted(set(labels.tolist()))
    tok_to_idx = {t: i for i, t in enumerate(unique_tokens)}
    n = len(unique_tokens)
    if n < 2:
        return float("nan"), float("nan")
    mat = np.zeros((n, n))
    idx = 0
    for length in seq_lengths:
        seq = labels[idx: idx + length]
        for t in range(length - 1):
            a, b = tok_to_idx[seq[t]], tok_to_idx[seq[t + 1]]
            mat[a, b] += 1
        idx += length
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    mat_norm = mat / row_sums
    entropies = [-(row[row > 0] * np.log(row[row > 0] + 1e-12)).sum() for row in mat_norm if row.sum() > 0]
    avg_entropy = float(np.mean(entropies)) if entropies else float("nan")
    baseline = np.log(n)
    return avg_entropy, baseline


train_mean = train_X.mean(dim=0, keepdim=True)
baseline_mse = F.mse_loss(train_mean.expand_as(val_X), val_X).item()
print(f"Baseline (predict train mean) held-out MSE: {baseline_mse:.5f}\n")


In [ ]:
score_rows = []

for name, result in sweep_results.items():
    cfg = result["config"]
    model = RVQModel(
        input_dim=128, code_dim=cfg["code_dim"],
        codebook_size=cfg["codebook_size"], n_levels=cfg["n_levels"],
    ).to(device)
    model.load_state_dict(result["state_dict"])
    model.eval()

    with torch.no_grad():
        val_recon, val_indices, _ = model(val_X)
        val_mse = F.mse_loss(val_recon, val_X).item()

        full_recon, full_indices, _ = model(final_embeddings.to(device))
        flat_labels = combine_indices(full_indices, cfg["codebook_size"]).cpu().numpy()

    effective_vocab = cfg["codebook_size"] ** cfg["n_levels"]
    n_used = len(set(flat_labels.tolist()))
    utilization = n_used / effective_vocab

    token_subject_spread = defaultdict(set)
    for tok, subj in zip(flat_labels, window_subjects):
        token_subject_spread[tok].add(subj)
    spread_counts = Counter(len(v) for v in token_subject_spread.values())
    frac_single_subject = spread_counts.get(1, 0) / max(len(token_subject_spread), 1)

    overlap_scores = []
    for (subj, stage), sessions in comparable_groups.items():
        rec_indices = [idx for _, idx in sessions]
        for i in range(len(rec_indices)):
            for j in range(i + 1, len(rec_indices)):
                dist_a = token_distribution_for_recording(rec_indices[i], sequence_lengths, flat_labels)
                dist_b = token_distribution_for_recording(rec_indices[j], sequence_lengths, flat_labels)
                overlap_scores.append(distribution_overlap(dist_a, dist_b))
    mean_overlap = float(np.mean(overlap_scores)) if overlap_scores else None

    trans_entropy, trans_baseline = transition_entropy_for_labels(flat_labels, sequence_lengths)

    score_rows.append({
        "config_name": name,
        "codebook_size": cfg["codebook_size"],
        "n_levels": cfg["n_levels"],
        "effective_vocab": effective_vocab,
        "codes_used": n_used,
        "utilization": utilization,
        "val_recon_mse": val_mse,
        "recon_improvement_over_baseline": 1 - (val_mse / baseline_mse),
        "frac_single_subject_tokens": frac_single_subject,
        "cross_session_overlap_mean": mean_overlap,
        "transition_entropy": trans_entropy,
        "transition_entropy_baseline": trans_baseline,
    })

    print(f"[{name}] vocab={effective_vocab:<6} used={n_used:<5} util={utilization:.1%} | "
          f"val_mse={val_mse:.5f} (baseline={baseline_mse:.5f}) | "
          f"1-subj frac={frac_single_subject:.1%} | sess overlap={mean_overlap}")

scores_df = pd.DataFrame(score_rows)


## 6. Funnel — eliminate broken configs before considering any BLM

Thresholds below are deliberately simple and conservative — the goal is to
catch *obviously* broken configs (collapsed codebook, no reconstruction
signal, subject-private vocabulary), not to pick a single winner yet. Adjust
thresholds if the printed numbers suggest they're too strict or too loose for
what this sweep actually produced.


In [ ]:
MIN_UTILIZATION = 0.10          # below this, codebook is collapsing
MIN_RECON_IMPROVEMENT = 0.05    # below this, barely beating the trivial mean-prediction baseline
MAX_SINGLE_SUBJECT_FRAC = 0.50  # above this, vocabulary is mostly subject-private, not reusable

def passes_funnel(row):
    reasons = []
    if row["utilization"] < MIN_UTILIZATION:
        reasons.append(f"low utilization ({row['utilization']:.1%} < {MIN_UTILIZATION:.0%})")
    if row["recon_improvement_over_baseline"] < MIN_RECON_IMPROVEMENT:
        reasons.append(f"weak reconstruction ({row['recon_improvement_over_baseline']:.1%} improvement)")
    if row["frac_single_subject_tokens"] > MAX_SINGLE_SUBJECT_FRAC:
        reasons.append(f"subject-private vocabulary ({row['frac_single_subject_tokens']:.1%} single-subject)")
    return len(reasons) == 0, reasons

scores_df["passes_funnel"] = False
scores_df["fail_reasons"] = ""
for i, row in scores_df.iterrows():
    ok, reasons = passes_funnel(row)
    scores_df.at[i, "passes_funnel"] = ok
    scores_df.at[i, "fail_reasons"] = "; ".join(reasons)

print("Funnel results:\n")
for _, row in scores_df.iterrows():
    status = "PASS" if row["passes_funnel"] else "FAIL"
    reason_str = f" — {row['fail_reasons']}" if row["fail_reasons"] else ""
    print(f"[{status}] {row['config_name']:<10}{reason_str}")

survivors = scores_df[scores_df["passes_funnel"]].copy()
print(f"\n{len(survivors)} / {len(scores_df)} configs passed the funnel.")


In [ ]:
# Rank survivors — recon quality and cross-subject generality matter most for
# a usable BLM vocabulary; utilization and transition structure as tiebreakers.
if len(survivors) > 0:
    survivors["rank_score"] = (
        survivors["recon_improvement_over_baseline"].rank(ascending=False)
        + (1 - survivors["frac_single_subject_tokens"]).rank(ascending=False)
        + survivors["utilization"].rank(ascending=False)
    )
    survivors = survivors.sort_values("rank_score")
    shortlist = survivors.head(3)
    print("Shortlist for BLM training (top 3 of the survivors):\n")
    print(shortlist[[
        "config_name", "effective_vocab", "utilization", "val_recon_mse",
        "recon_improvement_over_baseline", "frac_single_subject_tokens",
        "cross_session_overlap_mean",
    ]].to_string(index=False))
else:
    shortlist = survivors
    print("No configs passed the funnel — see Section 8 for what to try next "
          "before training any BLM.")


## 7. Comparison against GMM / HDBSCAN

Loads the clustering notebook's saved results (if present) so the full
picture — exploratory clustering vs every RVQ config — is visible in one
table.


In [ ]:
full_table = scores_df[[
    "config_name", "effective_vocab", "utilization", "val_recon_mse",
    "recon_improvement_over_baseline", "frac_single_subject_tokens",
    "cross_session_overlap_mean", "transition_entropy", "passes_funnel",
]].copy()
full_table = full_table.rename(columns={"config_name": "method"})

if os.path.exists(TOKEN_DISCOVERY_PATH):
    with open(TOKEN_DISCOVERY_PATH, "rb") as f:
        prior = pickle.load(f)
    gmm_k = prior.get("gmm_k_used")
    gmm_row = {
        "method": f"GMM (K={gmm_k})",
        "effective_vocab": gmm_k,
        "utilization": None,
        "val_recon_mse": None,
        "recon_improvement_over_baseline": None,
        "frac_single_subject_tokens": prior.get("frac_single_subject_tokens_gmm"),
        "cross_session_overlap_mean": (
            float(np.mean(prior["cross_session_overlap_scores_gmm"]))
            if prior.get("cross_session_overlap_scores_gmm") else None
        ),
        "transition_entropy": prior.get("gmm_transition_entropy"),
        "passes_funnel": None,
    }
    full_table = pd.concat([full_table, pd.DataFrame([gmm_row])], ignore_index=True)
else:
    print("No prior token_discovery_results.pkl found — showing RVQ sweep only.")

full_table


## 8. Save shortlist for the BLM stage

Saves the shortlisted configs' checkpoints and token sequences so the next
notebook (BLM training) can load them directly without retraining.


In [ ]:
if len(shortlist) > 0:
    shortlist_names = shortlist["config_name"].tolist()
    shortlist_payload = {}
    for name in shortlist_names:
        result = sweep_results[name]
        cfg = result["config"]
        model = RVQModel(
            input_dim=128, code_dim=cfg["code_dim"],
            codebook_size=cfg["codebook_size"], n_levels=cfg["n_levels"],
        ).to(device)
        model.load_state_dict(result["state_dict"])
        model.eval()
        with torch.no_grad():
            _, full_indices, _ = model(final_embeddings.to(device))
            flat_labels = combine_indices(full_indices, cfg["codebook_size"]).cpu().numpy()

        shortlist_payload[name] = {
            "config": cfg,
            "state_dict": result["state_dict"],
            "token_labels_flat": flat_labels,
        }

    out_path = os.path.join(RVQ_DIR, "shortlist_for_blm.pkl")
    with open(out_path, "wb") as f:
        pickle.dump({
            "shortlist_names": shortlist_names,
            "shortlist": shortlist_payload,
            "recording_names": recording_names,
            "sequence_lengths": sequence_lengths,
            "scores_table": scores_df.to_dict(orient="records"),
        }, f)
    print("Saved shortlist:", out_path)
    print("Configs going forward to BLM training:", shortlist_names)
else:
    print("Nothing to save — no configs passed the funnel. See below before "
          "training any BLM.")


## Reading the results — what to do next

- **If most/all configs failed the funnel**, don't train a BLM yet. On this
  little data (3684 windows), that most plausibly means the codebook/level
  combinations tried are too large for the data available — try smaller
  `codebook_size` values, or revisit whether `code_dim=32` is too high a
  bottleneck dimension for this few examples. This is exactly the outcome the
  funnel is meant to catch cheaply, versus finding it out after training 10 BLMs.
- **If 2-3 configs passed**, those are what's worth training a BLM against —
  compare *those* BLMs on held-out perplexity, not the full sweep.
- **If RVQ's best config doesn't clearly beat GMM/HDBSCAN** on cross-subject
  recurrence, that's a legitimate finding, not a failure — it means simple
  clustering was already good enough at this data size, and the real payoff
  of RVQ (a jointly-learned, more reconstructive vocabulary) may only show up
  once the full dataset and GPU access land.
